[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lugerlakes/signal_analysis_course/blob/main/Lab5_GMM_senales_desconocidas.ipynb)

<div style="border-left:6px solid #1F4E79;padding:10px 16px;background:#F3F7FB">

<span style="font-size:12px;letter-spacing:1px;color:#1F4E79">CURSO DE GUERRA ELECTRÓNICA · ASIGNATURA: ANÁLISIS DE SEÑALES DE RADAR</span>

# Laboratorio 5 · Señales desconocidas: clustering con GMM
### Separación automática de emisores, probabilidades de pertenencia y cotejo con la biblioteca

<span style="font-size:12px;color:#7a7a7a">MATERIAL DE INSTRUCCIÓN · DATOS SINTÉTICOS · Emisores y bibliotecas ficticios</span>
</div>

| | |
|---|---|
| ⏱️ **Duración estimada** | 3 a 4 horas |
| 📥 **Datos de entrada** | `D4_ESM_senales_desconocidas.xlsx` · `BIBLIOTECA_EMISORES_ficticia.xlsx` |
| 📤 **Producto al final** | Tabla de identificación (conocidos / ambiguos / desconocidos) + ficha del emisor desconocido |

**Objetivos de aprendizaje** — al terminar este laboratorio serás capaz de:

1. Explicar qué es el **aprendizaje no supervisado** y la idea de un **Modelo de Mezcla Gaussiana (GMM)**.
2. Preparar las variables (**estandarización**, logaritmo del PW) y justificar por qué.
3. Elegir el número de grupos con el criterio **BIC** (regla del codo).
4. Interpretar las **probabilidades de pertenencia** y la **verosimilitud** para detectar pulsos ambiguos y ruido.
5. Caracterizar cada grupo y **cotejarlo con una biblioteca de emisores**, asignando un nivel de confianza.
6. Tratar correctamente un **emisor desconocido** (no presente en la biblioteca).

**Cómo leer este cuaderno**

| Ícono | Significado |
|---|---|
| 📘 | Concepto o teoría breve |
| ▶️ | Celda de código para **ejecutar** (clic en la celda y `Shift + Enter`) |
| ✏️ | **Tu turno**: completa el código o responde |
| 💡 | Consejo práctico |
| ⚠️ | Error frecuente o precaución |
| 🎯 | Conexión con la Guerra Electrónica (ESM / ELINT) |

> Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla de forma extraña: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.

In [ ]:
# @title 🧰 Caja de herramientas (Labs 1–5) — ejecutar
# ------------------------------------------------------------------
# A. CARGA Y LIMPIEZA DE PDW (lo que aprendiste a mano en el Lab 1)
# ------------------------------------------------------------------
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4)

CARPETAS_DATOS = [".", "datos", "/content", "/content/datos", "/content/sample_data",
                  "/content/drive/MyDrive", "/content/drive/MyDrive/datos",
                  "/content/drive/MyDrive/Analisis_de_Señales"]


def buscar_archivo(nombre):
    """Busca 'nombre' en las carpetas habituales (local y Google Colab)."""
    for carpeta in CARPETAS_DATOS:
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    raise FileNotFoundError(
        f"No encontré '{nombre}'. Súbelo con el ícono de carpeta 📁 (panel izquierdo de Colab) "
        f"o copia su ruta exacta en la variable del archivo.")


def estandarizar_columnas(df):
    """Renombra las columnas de la exportación ESM a nombres estándar con unidades."""
    nuevos = {}
    for col in df.columns:
        c = str(col).upper()
        if c.startswith("UNNAMED") or c.strip() == "":
            continue
        if "TOA" in c:
            nuevos[col] = "TOA_us"
        elif "FREQ" in c or "FREC" in c:
            nuevos[col] = "FREQ_MHz"
        elif c.startswith("PW") or "WIDTH" in c:
            nuevos[col] = "PW_us"
        elif "AMP" in c:
            nuevos[col] = "AMP_dBm"
        elif "AOA" in c or "DOA" in c or "BEARING" in c:
            nuevos[col] = "AOA_deg"
        elif "PRI" in c:
            nuevos[col] = "PRI_us"
    columnas_vacias = [c for c in df.columns if c not in nuevos]
    return df.drop(columns=columnas_vacias).rename(columns=nuevos)


def limpiar_pdw(df, verbose=True):
    """Limpieza estándar de PDW: centinelas, vacíos, duplicados y orden por TOA."""
    n0 = len(df)
    df = df.copy()
    claves = [c for c in ["TOA_us", "FREQ_MHz", "PW_us"] if c in df.columns]
    df = df.dropna(subset=claves)
    n1 = len(df)
    malos = (df["FREQ_MHz"] <= 0) | (df["PW_us"] <= 0)      # registros "fantasma" del receptor
    df = df[~malos]
    n2 = len(df)
    df = df.drop_duplicates()
    n3 = len(df)
    df = df.sort_values("TOA_us").reset_index(drop=True)
    df["TOA_s"] = (df["TOA_us"] - df["TOA_us"].iloc[0]) / 1e6
    df["DTOA_us"] = df["TOA_us"].diff()
    if verbose:
        print(f"Filas originales ............ {n0:>7,}")
        print(f"  − con celdas vacías ....... {n0 - n1:>7,}")
        print(f"  − con ceros / centinela ... {n1 - n2:>7,}")
        print(f"  − duplicadas .............. {n2 - n3:>7,}")
        print(f"Filas válidas ............... {n3:>7,}")
    return df


# Carpeta web con los datos del curso (repositorio público). Déjala vacía ("") para trabajar sólo con archivos locales.
URL_DATOS = "https://raw.githubusercontent.com/lugerlakes/signal_analysis_course/main/datos/"


def asegurar_archivo(nombre):
    """Devuelve la ruta del archivo; si no está, lo descarga (URL_DATOS) o pide subirlo (Colab)."""
    try:
        return buscar_archivo(nombre)
    except FileNotFoundError:
        pass
    if URL_DATOS:
        import urllib.request, urllib.parse
        os.makedirs("datos", exist_ok=True)
        destino = os.path.join("datos", nombre)
        try:
            urllib.request.urlretrieve(URL_DATOS.rstrip("/") + "/" + urllib.parse.quote(nombre), destino)
            print(f"⬇️ '{nombre}' descargado desde el repositorio del curso")
            return destino
        except Exception as e:
            print(f"⚠️ No se pudo descargar '{nombre}' ({e}). Intentando otras opciones…")
            if os.path.exists(destino):
                os.remove(destino)
    try:
        from google.colab import files
        print(f"📁 No encuentro '{nombre}'. Selecciónalo desde tu computador:")
        subidos = files.upload()
        if nombre in subidos:
            return nombre
    except ImportError:
        pass
    return buscar_archivo(nombre)   # mensaje de error claro


def cargar_pdw(nombre_archivo, hoja=0, verbose=True):
    """Carga un archivo ESM (Excel o CSV), estandariza columnas y limpia."""
    ruta = asegurar_archivo(nombre_archivo)
    if ruta.lower().endswith(".csv"):
        crudo = pd.read_csv(ruta)
    else:
        crudo = pd.read_excel(ruta, sheet_name=hoja)
    df = limpiar_pdw(estandarizar_columnas(crudo), verbose=verbose)
    if verbose:
        dur = df["TOA_s"].iloc[-1]
        print(f"Duración de la captura ...... {dur:,.2f} s")
    return df
# ------------------------------------------------------------------
# B. ESTADÍSTICA DESCRIPTIVA (Lab 2)
# ------------------------------------------------------------------
def moda_continua(serie, bins="auto"):
    """Moda de una variable continua = centro de la barra más alta del histograma."""
    x = pd.Series(serie).dropna().values
    cuentas, bordes = np.histogram(x, bins=bins)
    i = np.argmax(cuentas)
    return 0.5 * (bordes[i] + bordes[i + 1])


def ic95_media(serie, confianza=0.95):
    """Intervalo de confianza para la MEDIA usando la distribución t de Student."""
    x = pd.Series(serie).dropna().values
    n = len(x)
    media = x.mean()
    ee = x.std(ddof=1) / np.sqrt(n)              # error estándar de la media
    t = stats.t.ppf(0.5 + confianza / 2, df=n - 1)
    return media - t * ee, media + t * ee


def ic95_mediana_bootstrap(serie, n_boot=2000, confianza=0.95, semilla=0):
    """Intervalo de confianza para la MEDIANA por remuestreo (bootstrap)."""
    x = pd.Series(serie).dropna().values
    rng = np.random.default_rng(semilla)
    medianas = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    a = (1 - confianza) / 2
    return np.quantile(medianas, a), np.quantile(medianas, 1 - a)


def resumen_estadistico(serie, nombre=None):
    """Tabla con los estadígrafos más usados en un informe de análisis."""
    x = pd.Series(serie).dropna()
    lo, hi = ic95_media(x)
    return pd.Series({
        "n": int(len(x)),
        "media": x.mean(),
        "IC95_media_inf": lo,
        "IC95_media_sup": hi,
        "mediana": x.median(),
        "moda (histograma)": moda_continua(x),
        "desv_estandar": x.std(ddof=1),
        "CV_%": 100 * x.std(ddof=1) / abs(x.mean()) if x.mean() != 0 else np.nan,
        "minimo": x.min(),
        "P5": x.quantile(0.05),
        "Q1 (P25)": x.quantile(0.25),
        "Q3 (P75)": x.quantile(0.75),
        "P95": x.quantile(0.95),
        "maximo": x.max(),
        "rango": x.max() - x.min(),
        "IQR": x.quantile(0.75) - x.quantile(0.25),
        "asimetria": stats.skew(x),
    }, name=nombre or getattr(serie, "name", "variable"))


def graficar_distribucion(serie, titulo="", unidad="", bins=60):
    """Histograma + KDE, boxplot y violín de una misma variable (3 vistas)."""
    x = pd.Series(serie).dropna()
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [2.2, 1, 1]})
    sns.histplot(x, bins=bins, kde=True, ax=ax[0], color="#4C72B0")
    ax[0].axvline(x.mean(), color="#C44E52", ls="--", lw=1.5, label=f"media = {x.mean():.4g}")
    ax[0].axvline(x.median(), color="#55A868", ls="-", lw=1.5, label=f"mediana = {x.median():.4g}")
    ax[0].legend(fontsize=9)
    ax[0].set_title(f"Histograma + KDE · {titulo}")
    ax[0].set_xlabel(unidad)
    sns.boxplot(y=x, ax=ax[1], color="#8172B2", width=0.4)
    ax[1].set_title("Boxplot"); ax[1].set_ylabel(unidad)
    sns.violinplot(y=x, ax=ax[2], color="#64B5CD", inner="quartile", cut=0)
    ax[2].set_title("Violín"); ax[2].set_ylabel(unidad)
    plt.tight_layout()
    plt.show()
# ------------------------------------------------------------------
# C. CARACTERIZACIÓN DE UN EMISOR (Lab 3)
# ------------------------------------------------------------------
from scipy.signal import find_peaks
from collections import Counter

C_LUZ = 299_792_458.0  # m/s


def banda_ieee(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.3, "VHF"), (1, "UHF"), (2, "L"), (4, "S"), (8, "C"), (12, "X"),
                   (18, "Ku"), (27, "K"), (40, "Ka"), (75, "V"), (110, "W")]:
        if f < lim:
            return b
    return "mm"


def banda_otan(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.25, "A"), (0.5, "B"), (1, "C"), (2, "D"), (3, "E"), (4, "F"), (6, "G"),
                   (8, "H"), (10, "I"), (20, "J"), (40, "K"), (60, "L"), (100, "M")]:
        if f < lim:
            return b
    return ">M"


def analizar_pri(toa_us, verbose=True):
    """Analiza la secuencia de llegadas de UN emisor y sugiere el tipo de PRI.

    Pasos: (1) DTOA = diferencias de TOA; (2) PRI base = valor más bajo frecuente;
    (3) se descartan huecos de iluminación y pulsos perdidos (DTOA > 1.8·PRI base);
    (4) se buscan niveles discretos y se mide la correlación pulso a pulso."""
    toa = np.sort(np.asarray(toa_us, float))
    d = np.diff(toa)
    d = d[d > 0]
    r = {"n_dtoa": len(d)}
    if len(d) < 10:
        r["tipo"] = "Insuficientes pulsos"
        return r
    # (2) PRI base sobre escala logarítmica (bins de 0.5 %)
    ld = np.log(d)
    lb = np.arange(ld.min(), np.quantile(ld, 0.98) + 0.01, 0.005)
    if len(lb) < 3:
        lb = np.linspace(ld.min(), ld.max() + 1e-6, 5)
    cnt, _ = np.histogram(ld, bins=lb)
    acum = np.convolve(cnt, np.ones(7), mode="same")
    i0 = int(np.argmax(acum >= 0.03 * len(d)))
    base = float(np.exp(lb[i0]))
    principal = (d >= 0.8 * base) & (d <= 1.8 * base)
    m = d[principal]
    r.update(pri_base=base, fraccion_principal=principal.mean(),
             fraccion_perdidos=(d > 1.8 * base).mean())
    med = np.median(m)
    r.update(pri_mediana=med, pri_media=m.mean(), pri_min=np.quantile(m, 0.01),
             pri_max=np.quantile(m, 0.99), pri_std=m.std(ddof=1),
             pri_cv_pct=100 * m.std(ddof=1) / m.mean())
    # (4a) niveles discretos
    ancho = max(0.003 * med, 0.05)
    bordes = np.arange(m.min() - ancho, m.max() + 2 * ancho, ancho)
    h, _ = np.histogram(m, bins=bordes)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    picos, _ = find_peaks(np.r_[0, h, 0], height=0.08 * h.max(), distance=3)
    picos = picos - 1
    niveles = []
    for p in picos:
        sel = np.abs(m - centros[p]) <= 1.5 * ancho
        if sel.sum():
            niveles.append(float(np.median(m[sel])))
    niveles = sorted(niveles)
    conc = np.mean(np.min(np.abs(m[:, None] - np.array(niveles)[None, :]), axis=1) <= 2 * ancho) if niveles else 0
    discreto = (1 <= len(niveles) <= 8) and conc >= 0.85
    # (4b) correlación entre PRI consecutivos (sólo pares sin pulsos perdidos)
    ok = principal[:-1] & principal[1:]
    a, b = d[:-1][ok], d[1:][ok]
    r1 = float(np.corrcoef(a, b)[0, 1]) if len(a) > 5 and a.std() > 0 and b.std() > 0 else 0.0
    r.update(niveles=niveles, concentracion=conc, autocorr_lag1=r1)
    # (5) clasificación
    if r["fraccion_principal"] < 0.4:
        r["tipo"] = "Sin estructura (posible ruido o mezcla)"
        r["detalle"] = (f"sólo {100 * r['fraccion_principal']:.0f} % de los DTOA forman un patrón: "
                        "no hay un tren de pulsos coherente")
        if verbose:
            print(f"Tipo de PRI sugerido : {r['tipo']}\nDetalle              : {r['detalle']}")
        return r
    if discreto and len(niveles) == 1:
        tipo = "Estable"
        detalle = f"PRI ≈ {niveles[0]:.2f} µs (PRF ≈ {1e6 / niveles[0]:.1f} Hz), CV = {r['pri_cv_pct']:.3f} %"
    elif discreto:
        niv = np.array(niveles)
        idx = np.argmin(np.abs(m[:, None] - niv[None, :]), axis=1)
        corridas = np.diff(np.flatnonzero(np.r_[True, idx[1:] != idx[:-1], True]))
        largo_corrida = corridas.mean()
        if largo_corrida >= 4:
            tipo = "Dwell & Switch"
            detalle = (f"{len(niv)} PRI: " + ", ".join(f"{v:.1f}" for v in niv) +
                       f" µs · ~{largo_corrida:.0f} pulsos por permanencia (dwell)")
            r["pulsos_por_dwell"] = largo_corrida
        else:
            tipo = f"Stagger {len(niv)} posiciones"
            k = len(niv)
            ventanas = [tuple(idx[i:i + k]) for i in range(len(idx) - k)]
            ventanas = [w for w in ventanas if len(set(w)) == k]
            seq = Counter(ventanas).most_common(1)[0][0] if ventanas else tuple(range(k))
            orden = [niv[j] for j in seq]
            r["secuencia"] = orden
            r["periodo_trama"] = float(niv.sum())
            detalle = ("secuencia " + " → ".join(f"{v:.1f}" for v in orden) +
                       f" µs · periodo de trama {niv.sum():.1f} µs · PRI media {niv.mean():.1f} µs")
    else:
        if r1 > 0.6:
            tipo = "Deslizante (sliding)"
            caidas = np.flatnonzero(np.diff(m) < -0.5 * (r["pri_max"] - r["pri_min"]))
            pulsos_rampa = float(np.median(np.diff(caidas))) if len(caidas) > 2 else np.nan
            r["pulsos_por_rampa"] = pulsos_rampa
            detalle = (f"rampa de {r['pri_min']:.0f} a {r['pri_max']:.0f} µs, ~{pulsos_rampa:.0f} pulsos por rampa")
        else:
            tipo = "Jitter (aleatorio)"
            pct = 100 * (r["pri_max"] - r["pri_min"]) / (2 * med)
            r["jitter_pct"] = pct
            detalle = f"PRI central ≈ {med:.1f} µs, variación ≈ ±{pct:.1f} %"
    r["tipo"] = tipo
    r["detalle"] = detalle
    if verbose:
        print(f"Tipo de PRI sugerido : {tipo}")
        print(f"Detalle              : {detalle}")
        print(f"Pulsos perdidos      : {100 * r['fraccion_perdidos']:.1f} % de los DTOA (múltiplos de la PRI o huecos de scan)")
    return r


def graficar_pri(toa_us, titulo="", max_pulsos=600):
    """Dos vistas clásicas: secuencia DTOA vs nº de pulso, e histograma de DTOA."""
    d = np.diff(np.sort(np.asarray(toa_us, float)))
    lim = np.quantile(d, 0.97) * 1.3
    fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
    n = min(len(d), max_pulsos)
    ax[0].plot(np.arange(n), d[:n], ".", ms=4)
    ax[0].set_ylim(0, lim)
    ax[0].set_xlabel("Nº de pulso"); ax[0].set_ylabel("DTOA (µs)")
    ax[0].set_title(f"Secuencia de PRI · {titulo}")
    ax[1].hist(d[d < lim], bins=200, color="#4C72B0")
    ax[1].set_xlabel("DTOA (µs)"); ax[1].set_ylabel("Nº de pulsos")
    ax[1].set_title("Histograma de DTOA")
    plt.tight_layout(); plt.show()


def detectar_iluminaciones(df, gap_us=100_000, min_pulsos=5):
    """Agrupa pulsos en 'iluminaciones' (ráfagas) separadas por huecos > gap_us.
    Se ignoran grupos con menos de 'min_pulsos' (pulsos sueltos de lóbulos laterales)."""
    toa = df["TOA_us"].values
    tsec = df["TOA_s"].values if "TOA_s" in df else toa / 1e6     # segundos desde el inicio de la captura
    amp = df["AMP_dBm"].values
    corte = np.r_[0, np.flatnonzero(np.diff(toa) > gap_us) + 1, len(toa)]
    filas = []
    for i, j in zip(corte[:-1], corte[1:]):
        if j - i < min_pulsos:
            continue
        t, a = toa[i:j], amp[i:j]
        sel = a >= a.max() - 3.0                       # pulsos dentro de 3 dB del máximo
        w = 10 ** (a[sel] / 10)
        # ancho del lóbulo principal: ajuste de una parábola (en dB) a los pulsos
        # que están a menos de 10 dB del máximo  →  ancho a -3 dB = 2·sqrt(3/|c2|)
        lob = a >= a.max() - 10.0
        tot = np.nan
        if lob.sum() >= 5:
            tc = (t[lob] - t[lob].mean()) / 1e3                      # ms
            c2 = np.polyfit(tc, a[lob], 2)[0]
            if c2 < 0:
                tot = 2 * np.sqrt(3.0 / -c2)
        ts = tsec[i:j]
        filas.append({
            "t_inicio_s": ts[0], "t_fin_s": ts[-1], "n_pulsos": j - i,
            "duracion_ms": (t[-1] - t[0]) / 1e3,
            "t_centro_s": np.sum(ts[sel] * w) / np.sum(w),
            "amp_max_dBm": a.max(),
            "tot_3dB_ms": tot,
        })
    return pd.DataFrame(filas)


def espectro_amplitud(df, fmax=200.0):
    """Espectro (FFT) de la amplitud: revela modulaciones periódicas (scan cónico)."""
    t = df["TOA_us"].values / 1e6
    a = df["AMP_dBm"].values
    fs = min(1.0 / np.median(np.diff(t)), 4 * fmax)
    tu = np.arange(t[0], t[-1], 1 / fs)
    au = np.interp(tu, t, a)
    au = (au - au.mean()) * np.hanning(len(au))
    esp = np.abs(np.fft.rfft(au)) / len(au)
    f = np.fft.rfftfreq(len(au), 1 / fs)
    sel = (f >= 2) & (f <= fmax)
    return f[sel], esp[sel]


def analizar_scan(df, gap_us=100_000, verbose=True):
    """Clasifica el tipo de exploración de antena a partir de AMP vs TOA."""
    il = detectar_iluminaciones(df, gap_us)
    r = {"n_iluminaciones": len(il)}
    durac_total = (df["TOA_us"].iloc[-1] - df["TOA_us"].iloc[0]) / 1e6
    if len(il) == 0:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="pulsos dispersos, sin iluminaciones definidas")
        if verbose:
            print(f"Tipo de scan sugerido : {r['tipo']}\nDetalle               : {r['detalle']}")
        return r
    if len(il) <= 2 and il["duracion_ms"].max() / 1e3 > 0.6 * durac_total:
        f, e = espectro_amplitud(df)
        k = int(np.argmax(e))
        prominencia = e[k] / (np.median(e) + 1e-12)
        prof = df["AMP_dBm"].quantile(0.95) - df["AMP_dBm"].quantile(0.05)
        if prominencia > 8 and prof > 1.5:
            r.update(tipo="Cónico", frecuencia_scan_Hz=float(f[k]), periodo_s=1 / float(f[k]),
                     profundidad_dB=float(prof),
                     detalle=f"modulación de amplitud a {f[k]:.1f} Hz (periodo {1000 / f[k]:.1f} ms), profundidad ≈ {prof:.1f} dB")
        else:
            r.update(tipo="Fijo (iluminación continua)", periodo_s=0.0,
                     detalle=f"amplitud estable (variación {prof:.1f} dB): posible SEGUIMIENTO / lock-on")
    elif len(il) == 2:
        dt2 = float(il["t_centro_s"].iloc[1] - il["t_centro_s"].iloc[0])
        r.update(tipo="Indeterminado (2 iluminaciones)", periodo_s=dt2,
                 detalle=f"sólo 2 iluminaciones separadas {dt2:.2f} s: periodo probable, BAJA confianza (captura más larga)")
    elif len(il) < 3:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="una sola iluminación: no se puede estimar el periodo")
    else:
        dt = np.diff(il["t_centro_s"].values)
        cv = dt.std() / dt.mean()
        if cv < 0.08:
            T = float(np.median(dt))
            tot = float(il["tot_3dB_ms"].median())
            haz = 360.0 * tot / 1000 / T
            r.update(tipo="Circular", periodo_s=T, rpm=60 / T, tot_3dB_ms=tot, haz_estimado_deg=haz,
                     detalle=f"periodo {T:.2f} s ({60 / T:.1f} rpm) · tiempo en blanco (3 dB) {tot:.1f} ms · haz ≈ {haz:.1f}°")
        else:
            pares = dt[:-1] + dt[1:]
            alterna = np.mean(np.sign(np.diff(dt)[:-1]) != np.sign(np.diff(dt)[1:])) > 0.7
            if alterna and pares.std() / pares.mean() < 0.08:
                T = float(np.median(pares))
                r.update(tipo="Sectorial (bidireccional)", periodo_s=T,
                         detalle=f"iluminaciones en pares alternos ({np.median(dt[::2]):.2f} s / {np.median(dt[1::2]):.2f} s) · periodo de barrido {T:.2f} s")
            else:
                r.update(tipo="Irregular", periodo_s=float(np.median(dt)),
                         detalle="intervalos irregulares: posible cambio de modo, exploración electrónica o varios emisores mezclados")
        largas = il["duracion_ms"] > 5 * il["duracion_ms"].median()
        if largas.any():
            r["detalle"] += f" · ⚠ {int(largas.sum())} iluminación(es) prolongada(s): posible paso a SEGUIMIENTO"
    if verbose:
        print(f"Tipo de scan sugerido : {r['tipo']}")
        print(f"Detalle               : {r['detalle']}")
    return r


def media_circular(angulos_deg):
    """Media de ángulos respetando el salto 359° → 0°."""
    a = np.deg2rad(np.asarray(angulos_deg, float))
    return float(np.rad2deg(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360)


def ficha_emisor(df, nombre="EMISOR", gap_us=100_000, verbose=True):
    """Ficha paramétrica completa de un emisor (RF, PW, PRI, scan, AOA)."""
    f = df["FREQ_MHz"]
    rango_rf = f.quantile(0.99) - f.quantile(0.01)
    agil = rango_rf > 15
    pri = analizar_pri(df["TOA_us"], verbose=False)
    scan = analizar_scan(df, gap_us=gap_us, verbose=False)
    ficha = {
        "EMISOR": nombre,
        "N_PULSOS": len(df),
        "T_INICIO_s": df["TOA_s"].min() if "TOA_s" in df else np.nan,
        "T_FIN_s": df["TOA_s"].max() if "TOA_s" in df else np.nan,
        "RF_MEDIANA_MHz": round(f.median(), 1),
        "RF_MIN_MHz": round(f.quantile(0.01), 1),
        "RF_MAX_MHz": round(f.quantile(0.99), 1),
        "RF_TIPO": "Ágil" if agil else "Fija",
        "BANDA_IEEE": banda_ieee(f.median()),
        "BANDA_OTAN": banda_otan(f.median()),
        "PW_MEDIANA_us": round(df["PW_us"].median(), 2),
        "PW_P5_us": round(df["PW_us"].quantile(0.05), 2),
        "PW_P95_us": round(df["PW_us"].quantile(0.95), 2),
        "PRI_TIPO": pri.get("tipo"),
        "PRI_DETALLE": pri.get("detalle"),
        "PRI_MIN_us": round(pri.get("pri_min", np.nan), 1),
        "PRI_MAX_us": round(pri.get("pri_max", np.nan), 1),
        "PRI_MEDIA_us": round(pri.get("pri_media", np.nan), 1),
        "PRF_MEDIA_Hz": round(1e6 / pri.get("pri_media", np.nan), 1),
        "SCAN_TIPO": scan.get("tipo"),
        "SCAN_PERIODO_s": round(scan.get("periodo_s", np.nan), 3),
        "SCAN_DETALLE": scan.get("detalle"),
        "AOA_MEDIA_deg": round(media_circular(df["AOA_deg"]), 1) if "AOA_deg" in df else np.nan,
        "AMP_MAX_dBm": round(df["AMP_dBm"].max(), 1),
    }
    if verbose:
        for k, v in ficha.items():
            print(f"{k:<16}: {v}")
    return ficha
# ------------------------------------------------------------------
# D. DEINTERLEAVING (Lab 4)
# ------------------------------------------------------------------
def asignar_por_ventanas(df, ventanas, etiqueta="GRUPO"):
    """Asigna cada pulso al primer grupo cuyas ventanas (mín, máx) contengan sus parámetros."""
    df = df.copy()
    df[etiqueta] = "SIN_ASIGNAR"
    for grupo, limites in ventanas.items():
        sel = df[etiqueta] == "SIN_ASIGNAR"
        for col, (lo, hi) in limites.items():
            sel &= df[col].between(lo, hi)
        df.loc[sel, etiqueta] = grupo
    return df


def histograma_diferencias(toa_us, orden_max=4, max_us=3000, bins=600, titulo="", log=False):
    """Histograma de diferencias de TOA de orden 1..k (idea base de CDIF/SDIF)."""
    toa = np.sort(np.asarray(toa_us, float))
    fig, ax = plt.subplots(orden_max, 1, figsize=(13, 2.0 * orden_max), sharex=True)
    bordes = np.linspace(0, max_us, bins + 1)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    for k in range(1, orden_max + 1):
        d = toa[k:] - toa[:-k]
        h, _ = np.histogram(d, bins=bordes)
        ax[k - 1].fill_between(centros, h + (1 if log else 0), step="mid", color="#4C72B0")
        if log:
            ax[k - 1].set_yscale("log")
        ax[k - 1].set_ylabel(f"orden {k}")
    ax[0].set_title(f"Histogramas de diferencias de TOA · {titulo}")
    ax[-1].set_xlabel("Diferencia de TOA (µs)")
    plt.tight_layout(); plt.show()
# ------------------------------------------------------------------
# E. COTEJO CON BIBLIOTECA DE EMISORES (Lab 5)
# ------------------------------------------------------------------
PESOS = {"RF": 0.35, "PRI": 0.30, "PW": 0.20, "SCAN": 0.15}


def _dentro(valor, lo, hi, tol_rel=0.03):
    margen = tol_rel * max(abs(lo), abs(hi), 1e-9)
    return (lo - margen) <= valor <= (hi + margen)


def comparar_con_biblioteca(ficha, biblioteca, top=3):
    """Compara una ficha de emisor contra cada entrada de la biblioteca.

    Puntaje (0 a 1) = suma de pesos de los parámetros que calzan:
    RF 0.35 · PRI 0.30 · PW 0.20 · SCAN 0.15
    La RF es criterio EXCLUYENTE: si la RF no calza, el puntaje es 0."""
    filas = []
    for _, b in biblioteca.iterrows():
        rf_ok = _dentro(ficha["RF_MEDIANA_MHz"], b["RF_MIN_MHz"], b["RF_MAX_MHz"], 0.005)
        pri_ok = _dentro(ficha["PRI_MEDIA_us"], b["PRI_MIN_us"], b["PRI_MAX_us"])
        pw_ok = _dentro(ficha["PW_MEDIANA_us"], b["PW_MIN_us"], b["PW_MAX_us"], 0.10)
        s_lib = str(b["SCAN_TIPO"]).lower()
        s_obs = str(ficha["SCAN_TIPO"]).split(" ")[0].lower()
        scan_ok = s_obs[:5] in s_lib
        puntaje = (PESOS["RF"] * rf_ok + PESOS["PRI"] * pri_ok + PESOS["PW"] * pw_ok + PESOS["SCAN"] * scan_ok)
        if not rf_ok:
            puntaje = 0.0
        filas.append({"ID_LIB": b["ID_LIB"], "DESIGNACION": b["DESIGNACION"], "FUNCION": b["FUNCION"],
                      "RF": "✔" if rf_ok else "✘", "PRI": "✔" if pri_ok else "✘",
                      "PW": "✔" if pw_ok else "✘", "SCAN": "✔" if scan_ok else "✘",
                      "PUNTAJE": round(puntaje, 2), "AMENAZA": b["AMENAZA"]})
    res = pd.DataFrame(filas).sort_values("PUNTAJE", ascending=False).head(top).reset_index(drop=True)
    return res


def elegir_k_por_bic(bics, ks, tolerancia=0.02):
    """Regla del 'codo': el K más pequeño cuyo BIC está a menos del 2 % del
    recorrido total (máx − mín) respecto del mejor BIC."""
    bics = np.asarray(bics, float)
    umbral = bics.min() + tolerancia * (bics.max() - bics.min())
    return int(np.asarray(ks)[np.argmax(bics <= umbral)])


def preparar_variables(df, columnas=("FREQ_MHz", "PW_us", "AOA_deg")):
    """Matriz de variables para clustering: log10 del PW y estandarización (z-score)."""
    from sklearn.preprocessing import StandardScaler
    X = df[list(columnas)].copy()
    if "PW_us" in X:
        X["PW_us"] = np.log10(X["PW_us"])
    return StandardScaler().fit_transform(X.values)


def ajustar_gmm(df, columnas=("FREQ_MHz", "PW_us", "AOA_deg"), k_max=12, n_muestra=8000, semilla=0, graficar=True):
    """Ajusta GMM eligiendo K por BIC (regla del codo). Devuelve (modelo, X, K)."""
    from sklearn.mixture import GaussianMixture
    X = preparar_variables(df, columnas)
    rng = np.random.default_rng(semilla)
    idx = rng.choice(len(X), min(n_muestra, len(X)), replace=False)
    ks = list(range(1, k_max + 1))
    bics = [GaussianMixture(k, covariance_type="full", n_init=2, random_state=semilla).fit(X[idx]).bic(X[idx])
            for k in ks]
    K = elegir_k_por_bic(bics, ks)
    if graficar:
        plt.figure(figsize=(8, 3.2))
        plt.plot(ks, bics, "o-"); plt.axvline(K, color="#C44E52", ls="--", label=f"K elegido = {K}")
        plt.xlabel("Nº de componentes K"); plt.ylabel("BIC (menor es mejor)"); plt.legend()
        plt.title("Selección del número de grupos por BIC"); plt.tight_layout(); plt.show()
    gmm = GaussianMixture(K, covariance_type="full", n_init=3, random_state=semilla).fit(X)
    return gmm, X, K


def nivel_confianza(candidatos):
    """Traduce el puntaje del mejor candidato a un nivel de confianza de identificación."""
    p1 = round(float(candidatos.loc[0, "PUNTAJE"]), 2)
    p2 = round(float(candidatos.loc[1, "PUNTAJE"]), 2) if len(candidatos) > 1 else 0.0
    if p1 >= 0.85 and round(p1 - p2, 2) >= 0.15:
        return "ALTA"
    if p1 >= 0.85:
        return "MEDIA (candidatos cercanos)"
    if p1 >= 0.65:
        return "BAJA"
    return "SIN CORRELACIÓN → DESCONOCIDO"
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
print('Caja de herramientas lista ✔')

---
## 1. Del analista al algoritmo

En el Lab 4 separamos emisores **dibujando ventanas a mano**. Funciona con 5 emisores… pero no con 30, no entrega una medida de **confianza** y depende del criterio de cada analista.

📘 **Aprendizaje no supervisado (clustering):** el algoritmo recibe los pulsos **sin etiquetas** ("este pulso es del emisor X") y los agrupa por **similitud**. Nosotros después interpretamos qué es cada grupo.

| Método | Idea | Forma de los grupos | ¿Da probabilidades? | Comentario |
|---|---|---|---|---|
| K-means | Cada pulso al centro más cercano | Esferas del mismo tamaño | No (asignación "dura") | Simple, rápido, sensible a escalas |
| **GMM** | Los datos son una **suma de campanas de Gauss** | **Elipses** de cualquier tamaño y orientación | **Sí** (asignación "blanda") | Permite medir confianza y detectar ruido |
| DBSCAN / HDBSCAN | Grupos = zonas densas | Cualquier forma | No (marca ruido) | No requiere K; sensible a parámetros de densidad |

---
## 2. La idea del GMM en una dimensión

📘 Un **Modelo de Mezcla Gaussiana** supone que el histograma de los datos es la **suma de K campanas** (distribuciones normales). Cada campana *k* tiene:
* un **peso** $\pi_k$ (qué fracción de los pulsos le pertenece),
* una **media** $\mu_k$ (dónde está),
* una **desviación estándar** $\sigma_k$ (o, en varias dimensiones, una **matriz de covarianza** que da forma a la elipse).

Ejemplo con datos sintéticos: PW de tres emisores mezclados.

In [ ]:
from scipy.stats import norm
rng = np.random.default_rng(4)
pw_mix = np.r_[rng.normal(0.5, 0.05, 600), rng.normal(1.0, 0.08, 300), rng.normal(1.6, 0.06, 100)].reshape(-1, 1)

gmm1d = GaussianMixture(n_components=3, random_state=0).fit(pw_mix)
orden1d = np.argsort(gmm1d.means_.ravel())          # campanas ordenadas de menor a mayor PW
x = np.linspace(0.2, 1.9, 500)
plt.figure(figsize=(12, 4))
plt.hist(pw_mix, bins=120, density=True, color="lightgray", label="datos (histograma)")
total = np.zeros_like(x)
for k in orden1d:
    pk, mk, sk = gmm1d.weights_[k], gmm1d.means_[k, 0], np.sqrt(gmm1d.covariances_[k, 0, 0])
    curva = pk * norm.pdf(x, mk, sk); total += curva
    plt.plot(x, curva, lw=2, label=f"campana {k}: peso {pk:.2f} · media {mk:.2f} µs · σ {sk:.3f}")
plt.plot(x, total, "k--", lw=1.5, label="suma (modelo)")
plt.xlabel("PW (µs)"); plt.ylabel("densidad"); plt.legend(fontsize=9); plt.title("GMM con 3 componentes")
plt.show()

### 2.1 Probabilidad de pertenencia
📘 Para cada pulso el GMM calcula la **probabilidad de pertenecer a cada campana**. Un pulso justo entre dos campanas tendrá probabilidades repartidas (p. ej. 0,55 / 0,45): es un pulso **ambiguo**.

In [ ]:
xs = np.linspace(0.5, 1.0, 501).reshape(-1, 1)
p_xs = gmm1d.predict_proba(xs)[:, orden1d]
frontera = xs[np.argmin(np.abs(p_xs[:, 0] - p_xs[:, 1])), 0]     # donde ambas campanas empatan
print(f"Frontera entre la campana de 0,5 µs y la de 1,0 µs ≈ {frontera:.3f} µs\n")
print("   PW (µs)   P(≈0,5 µs)  P(≈1,0 µs)  P(≈1,6 µs)")
for v in [0.50, frontera - 0.03, frontera, frontera + 0.03, 1.00, 1.60]:
    p = gmm1d.predict_proba([[v]])[0][orden1d]
    print(f"   {v:6.3f}    {p[0]:9.3f}   {p[1]:9.3f}   {p[2]:9.3f}")

### 2.2 ¿Cómo encuentra las campanas? El algoritmo EM
📘 El GMM se ajusta con el algoritmo **EM** (*Expectation–Maximization*), que repite dos pasos hasta converger:
1. **E (esperanza):** con las campanas actuales, calcula para cada pulso la probabilidad de pertenecer a cada una.
2. **M (maximización):** con esas probabilidades, recalcula peso, media y dispersión de cada campana.

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(18, 6), sharey=True)
for fila, semilla, rotulo in [(0, 2, "arranque A"), (1, 1, "arranque B")]:
    for a, it in zip(ax[fila], [1, 3, 8, 100]):
        g = GaussianMixture(3, max_iter=it, init_params="random_from_data", random_state=semilla).fit(pw_mix)
        a.hist(pw_mix, bins=120, density=True, color="lightgray")
        for k in range(3):
            a.plot(x, g.weights_[k] * norm.pdf(x, g.means_[k, 0], np.sqrt(g.covariances_[k, 0, 0])), lw=2)
        a.set_title(f"{rotulo} · {it} iteración(es)"); a.set_xlabel("PW (µs)")
plt.tight_layout(); plt.show()

🎯 **Arriba:** las campanas se van desplazando iteración a iteración hasta calzar con los tres grupos. **Abajo:** con otro punto de partida el algoritmo puede quedar **atrapado** en una solución peor (por ejemplo, dos campanas compartiendo el grupo de 0,5 µs y una sola cubriendo 1,0 y 1,6 µs). Por eso se usa `n_init=3` (o más): se ajusta varias veces desde arranques distintos y se conserva el mejor resultado.

---
## 3. Datos del escenario

In [ ]:
df = cargar_pdw("D4_ESM_senales_desconocidas.xlsx")
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
ax[0].scatter(df["AOA_deg"], df["FREQ_MHz"], s=1, alpha=0.3); ax[0].set_xlabel("AOA (°)"); ax[0].set_ylabel("RF (MHz)")
ax[1].scatter(df["FREQ_MHz"], df["PW_us"], s=1, alpha=0.3); ax[1].set_yscale("log")
ax[1].set_xlabel("RF (MHz)"); ax[1].set_ylabel("PW (µs, log)")
plt.tight_layout(); plt.show()

---
## 4. Preparar las variables

### 4.1 ¿Qué variables usar?
🎯 Usamos los parámetros **propios de cada pulso**: **RF, PW y AOA**. **No** usamos el DTOA: en una mezcla no tiene significado (Lab 4). La amplitud tampoco (depende de la distancia y del scan).

### 4.2 ¿Por qué estandarizar?
Mira las escalas: la RF varía en **miles** de MHz, el AOA en **cientos** de grados y el PW en **décimas** de µs.

In [ ]:
df[["FREQ_MHz", "PW_us", "AOA_deg"]].describe().loc[["min", "max", "std"]].round(2)

Un algoritmo basado en **distancias** (como K-means) "ve" casi sólo la RF. Experimento: K-means con 7 grupos **sin** estandarizar:

In [ ]:
X_crudo = df[["FREQ_MHz", "PW_us", "AOA_deg"]].values
km = KMeans(7, n_init=5, random_state=0).fit(X_crudo)
df["KMEANS_CRUDO"] = km.labels_
plt.figure(figsize=(12, 4.5))
plt.scatter(df["AOA_deg"], df["FREQ_MHz"], c=df["KMEANS_CRUDO"], s=2, cmap="tab10")
plt.xlabel("AOA (°)"); plt.ylabel("RF (MHz)"); plt.title("K-means SIN estandarizar: observa el grupo en ≈ 9400 MHz")
plt.colorbar(label="grupo"); plt.show()

# ¿Cómo repartió K-means los pulsos de ≈ 9400 MHz según su dirección de llegada?
x9 = df[df["FREQ_MHz"].between(9250, 9550)]
pd.crosstab(x9["KMEANS_CRUDO"], (x9["AOA_deg"] // 10 * 10).astype(int).rename("AOA (décadas de grados)")).loc[:, lambda t: t.sum() > 50]

⚠️ En la tabla, **un mismo grupo** de K-means contiene pulsos de ≈ 9400 MHz que llegan desde ≈ 120° **y** desde ≈ 250°: son dos emisores distintos, pero como su RF es parecida, K-means los fusionó. Mientras tanto, "gasta" grupos en partir el ruido en franjas de RF (los colores del fondo del gráfico).

📘 **Estandarizar (z-score):** a cada variable se le resta su media y se divide por su desviación estándar: $z = (x - \bar x)/s$. Así todas quedan en la misma escala (media 0, desviación 1).

💡 Además usamos el **logaritmo del PW**: los PW van de 0,1 a 20 µs (dos órdenes de magnitud) y en escala log las nubes quedan de tamaño parecido.

⚠️ **AOA cerca de 0°/360°:** un emisor en 359° y otro en 1° están a 2° pero numéricamente a 358. Si tienes emisores cerca del norte, usa $\sin(AOA)$ y $\cos(AOA)$ como dos variables. En este escenario no ocurre.

In [ ]:
X = preparar_variables(df, ("FREQ_MHz", "PW_us", "AOA_deg"))     # log10(PW) + z-score
print("Forma de la matriz:", X.shape)
print("Media por columna   :", X.mean(axis=0).round(3))
print("Desv. est. por col. :", X.std(axis=0).round(3))

---
## 5. ¿Cuántos grupos? El criterio BIC

📘 Con más componentes el modelo siempre se ajusta "mejor", pero se arriesga a **sobreajustar** (inventar grupos). El **BIC** (*Bayesian Information Criterion*) equilibra:

$$BIC = \underbrace{-2\ln(\text{verosimilitud})}_{\text{qué tan mal ajusta}} + \underbrace{p\,\ln(n)}_{\text{castigo por complejidad}}$$

**Menor BIC = mejor compromiso.** En la práctica el BIC suele bajar fuerte y luego quedar casi plano: se elige el **codo**, el K más pequeño que ya está prácticamente en el mínimo.

▶️ Ajustamos GMM con K = 1 … 12 sobre una muestra de 8000 pulsos (para que sea rápido):

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(X), 8000, replace=False)
ks = list(range(1, 13))
bics = []
for k in ks:
    g = GaussianMixture(n_components=k, covariance_type="full", n_init=2, random_state=0).fit(X[idx])
    bics.append(g.bic(X[idx]))
    print(f"K = {k:2d} · BIC = {bics[-1]:12,.0f}")

K = elegir_k_por_bic(bics, ks)
plt.figure(figsize=(9, 3.5))
plt.plot(ks, bics, "o-"); plt.axvline(K, color="#C44E52", ls="--", label=f"codo: K = {K}")
plt.xlabel("K (nº de componentes)"); plt.ylabel("BIC"); plt.legend(); plt.title("Selección de K por BIC"); plt.show()

---
## 6. Ajustar el GMM y leer sus resultados

In [ ]:
gmm = GaussianMixture(n_components=K, covariance_type="full", n_init=3, random_state=0).fit(X)

etiquetas = gmm.predict(X)                       # grupo más probable de cada pulso
probas = gmm.predict_proba(X)                    # matriz n_pulsos × K de probabilidades
df["P_MAX"] = probas.max(axis=1)                 # confianza de la asignación
df["LOGVEROS"] = gmm.score_samples(X)            # qué tan "típico" es el pulso para el modelo

# Renombramos los grupos C1, C2, … ordenados por RF (así los nombres no dependen del azar)
orden = pd.Series(df["FREQ_MHz"].values).groupby(etiquetas).median().sort_values().index
mapa = {k: f"C{i+1}" for i, k in enumerate(orden)}
df["CLUSTER"] = [mapa[e] for e in etiquetas]
df["CLUSTER"].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 5.5))
for c, sub in df.groupby("CLUSTER"):
    ax[0].scatter(sub["AOA_deg"], sub["FREQ_MHz"], s=2, label=c)
    ax[1].scatter(sub["FREQ_MHz"], sub["PW_us"], s=2, label=c)
ax[0].set_xlabel("AOA (°)"); ax[0].set_ylabel("RF (MHz)"); ax[0].legend(markerscale=5)
ax[1].set_yscale("log"); ax[1].set_xlabel("RF (MHz)"); ax[1].set_ylabel("PW (µs, log)")
fig.suptitle("Grupos encontrados por el GMM"); plt.tight_layout(); plt.show()

### 6.1 Vista 3D (opcional)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401
fig = plt.figure(figsize=(10, 7))
ax3 = fig.add_subplot(projection="3d")
for c, sub in df.groupby("CLUSTER"):
    s = sub.sample(min(len(sub), 1500), random_state=0)
    ax3.scatter(s["FREQ_MHz"], s["AOA_deg"], np.log10(s["PW_us"]), s=3, label=c)
ax3.set_xlabel("RF (MHz)"); ax3.set_ylabel("AOA (°)"); ax3.set_zlabel("log10 PW")
ax3.legend(markerscale=4); plt.show()

# Versión interactiva (gira con el mouse) si plotly está disponible (lo está en Colab):
try:
    import plotly.express as px
    s = df.sample(6000, random_state=0)
    px.scatter_3d(s, x="FREQ_MHz", y="AOA_deg", z=np.log10(s["PW_us"]), color="CLUSTER",
                  labels={"z": "log10 PW"}, height=650, opacity=0.6).update_traces(marker_size=2).show()
except ImportError:
    print("(plotly no instalado: se omite la vista interactiva)")

### 6.2 Confianza de cada asignación
📘 `P_MAX` es la probabilidad del grupo asignado. Si es menor que, por ejemplo, 0,95, el pulso es **ambiguo** (podría ser de dos grupos).

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.hist(df["P_MAX"], bins=50); plt.yscale("log")
plt.xlabel("Probabilidad del grupo asignado"); plt.ylabel("Nº de pulsos (log)"); plt.title("Confianza de la asignación")
plt.show()
ambiguos = df["P_MAX"] < 0.95
print(f"Pulsos ambiguos (P < 0,95): {ambiguos.sum()} ({100*ambiguos.mean():.2f} %)")
df[ambiguos].groupby("CLUSTER").size()

### 6.3 Verosimilitud: ¿qué tan "normal" es cada pulso?
📘 `score_samples` mide qué tan probable es un pulso **según el modelo completo**. Los pulsos con verosimilitud muy baja no se parecen a ningún grupo compacto: candidatos a **ruido**.

In [ ]:
umbral = df["LOGVEROS"].quantile(0.03)
raros = df["LOGVEROS"] < umbral
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].hist(df["LOGVEROS"], bins=100); ax[0].axvline(umbral, color="r", ls="--", label="percentil 3")
ax[0].set_xlabel("log-verosimilitud"); ax[0].legend()
ax[1].scatter(df.loc[~raros, "AOA_deg"], df.loc[~raros, "FREQ_MHz"], s=1, color="lightgray", label="típicos")
ax[1].scatter(df.loc[raros, "AOA_deg"], df.loc[raros, "FREQ_MHz"], s=4, color="r", label="baja verosimilitud")
ax[1].set_xlabel("AOA (°)"); ax[1].set_ylabel("RF (MHz)"); ax[1].legend(markerscale=4)
plt.tight_layout(); plt.show()

---
## 7. Caracterizar cada grupo
Aplicamos la ficha del Lab 3 a cada grupo. Un grupo cuya PRI **no tiene estructura** es ruido.

In [ ]:
fichas = [ficha_emisor(sub, c, verbose=False) for c, sub in df.groupby("CLUSTER")]
tabla = pd.DataFrame(fichas).set_index("EMISOR")
tabla[["N_PULSOS", "T_INICIO_s", "T_FIN_s", "RF_MEDIANA_MHz", "RF_MIN_MHz", "RF_MAX_MHz", "RF_TIPO", "PW_MEDIANA_us",
       "AOA_MEDIA_deg", "PRI_TIPO", "PRI_DETALLE", "SCAN_TIPO", "SCAN_DETALLE"]]

In [ ]:
es_ruido = tabla["PRI_TIPO"].str.startswith("Sin estructura")
print("Grupos clasificados como RUIDO:", list(tabla.index[es_ruido]))
emisores = tabla[~es_ruido].copy()
print(f"Emisores válidos: {len(emisores)}")

### ✏️ Tu turno 1
1. ¿Qué grupo tiene **agilidad en frecuencia**? ¿Cuántos MHz abarca? ¿El GMM lo separó en una sola componente o en varias?
2. ¿Qué grupo aparece **sólo durante unos segundos**? ¿Qué características tiene y por qué preocupa?
3. ¿Qué grupo tiene un scan **cónico**?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. El grupo en ≈ 9300–9500 MHz (AOA ≈ 122°, PW ≈ 0,9 µs): ≈ 200 MHz de agilidad; con covarianza "completa" el GMM lo modela como **una elipse alargada** (una sola componente). Si viera varias componentes para un mismo emisor ágil, el analista debe **fusionarlas** (mismo AOA, mismo PW, misma PRI).
2. El de ≈ 9400 MHz, AOA ≈ 250°, entre ≈ 30 y 31,5 s: PRF alta (≈ 10 kHz), **Dwell & Switch** y **iluminación fija** → radar pulso-Doppler **en seguimiento**: alta amenaza.
3. El de banda **Ku** (≈ 16 GHz, AOA ≈ 185°), con modulación de amplitud a ≈ 25 Hz.


</details>

---
## 8. Cotejo con la biblioteca de emisores

📘 Una **biblioteca de emisores** (base de datos ELINT / *EOB*) contiene los rangos de parámetros de los radares conocidos. Identificar = encontrar la entrada cuyos rangos contengan lo medido. *(La biblioteca de este curso es **ficticia**.)*

In [ ]:
bib = pd.read_excel(asegurar_archivo("BIBLIOTECA_EMISORES_ficticia.xlsx"))
bib

### 8.1 Regla de puntaje
`comparar_con_biblioteca` revisa, para cada entrada, si la RF, la PRI media, el PW y el tipo de scan medidos caen dentro de sus rangos (con una pequeña tolerancia) y suma pesos:

| Parámetro | Peso | Nota |
|---|---|---|
| RF | 0,35 | **Criterio excluyente**: si no calza, puntaje 0 |
| PRI | 0,30 | PRI media dentro de [mín, máx] |
| PW | 0,20 | Mediana dentro de [mín, máx] |
| Scan | 0,15 | Tipo compatible |

| Mejor puntaje | Diferencia con el 2º | Nivel de confianza |
|---|---|---|
| ≥ 0,85 | ≥ 0,15 | **ALTA** |
| ≥ 0,85 | < 0,15 | **MEDIA** (candidatos cercanos) |
| 0,65 – 0,85 | — | **BAJA** |
| < 0,65 | — | **SIN CORRELACIÓN → DESCONOCIDO** |

Ejemplo con el grupo de banda Ku:

In [ ]:
ku = emisores[emisores["BANDA_IEEE"] == "Ku"].iloc[0]
cand = comparar_con_biblioteca(ku, bib)
display(cand)
print("Nivel de confianza:", nivel_confianza(cand))

🎯 Dos entradas calzan en RF, PRI y PW (LIB-09 y LIB-10). **El tipo de scan (cónico) desempata.** Esto muestra por qué una buena caracterización del scan vale tanto como la RF.

### 8.2 Todos los grupos

In [ ]:
filas = []
for nombre, f in emisores.iterrows():
    f = f.copy(); f["EMISOR"] = nombre
    cand = comparar_con_biblioteca(f, bib)
    conf = nivel_confianza(cand)
    desconocido = conf.startswith("SIN")
    filas.append({
        "GRUPO": nombre, "RF_MHz": f["RF_MEDIANA_MHz"], "AOA_deg": f["AOA_MEDIA_deg"], "PW_us": f["PW_MEDIANA_us"],
        "PRI": f["PRI_TIPO"], "PRF_Hz": f["PRF_MEDIA_Hz"], "SCAN": f["SCAN_TIPO"],
        "MEJOR_CANDIDATO": "—" if desconocido else f"{cand.loc[0, 'ID_LIB']} {cand.loc[0, 'DESIGNACION']}",
        "FUNCION": "—" if desconocido else cand.loc[0, "FUNCION"],
        "PUNTAJE": cand.loc[0, "PUNTAJE"], "CONFIANZA": conf,
        "AMENAZA_LIB": "—" if desconocido else cand.loc[0, "AMENAZA"],
    })
identificacion = pd.DataFrame(filas)
identificacion

### 8.3 El emisor desconocido
🎯 Un emisor que no calza con la biblioteca **no es un error**: puede ser un radar nuevo, uno modificado, un modo no catalogado o un error de la biblioteca. Es **el hallazgo más valioso** de una misión ELINT. Procedimiento:
1. Asignarle una **designación provisoria** (p. ej. `DESC-01`).
2. Documentar **todos** sus parámetros con su incertidumbre y las fechas/horas de detección.
3. Revisar si se parece a alguna entrada (¿un modo nuevo de un radar conocido?).
4. Recomendar **colección adicional** y su envío al centro de análisis ELINT para actualizar la biblioteca.

In [ ]:
desc = identificacion[identificacion["CONFIANZA"].str.startswith("SIN")]
for _, fila in desc.iterrows():
    f = emisores.loc[fila["GRUPO"]].copy(); f["EMISOR"] = fila["GRUPO"]
    print("=" * 78)
    print(f" EMISOR DESCONOCIDO · designación provisoria DESC-01 (grupo {fila['GRUPO']})")
    print("=" * 78)
    for k in ["N_PULSOS", "RF_MEDIANA_MHz", "RF_MIN_MHz", "RF_MAX_MHz", "BANDA_IEEE", "BANDA_OTAN", "PW_MEDIANA_us",
              "PW_P5_us", "PW_P95_us", "PRI_TIPO", "PRI_DETALLE", "PRF_MEDIA_Hz", "SCAN_TIPO", "SCAN_DETALLE", "AOA_MEDIA_deg"]:
        print(f" {k:<16}: {f[k]}")
    print("\n Candidatos más cercanos de la biblioteca (para referencia):")
    display(comparar_con_biblioteca(f, bib))

### ✏️ Tu turno 2 · Sensibilidad
Repite el ajuste forzando **K = 5** y luego **K = 10** (cambia `K` y vuelve a ejecutar desde la sección 6). Describe qué ocurre en cada caso. ¿Qué te enseña sobre la elección de K?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


* **K = 5:** el modelo se queda corto: **fusiona** dos emisores en una sola componente (o absorbe al ruido dentro de un emisor), y la ficha de ese grupo muestra una PRI "rara" o mezclada.
* **K = 10:** sobran componentes: el ruido se reparte en varias, o un emisor (típicamente el ágil o el de muchos pulsos) se **parte** en dos o más grupos con el mismo AOA, PW y PRI → el analista debe **fusionarlos**.

**Lección:** el BIC orienta, pero la decisión final la valida el **analista** con la física: un grupo = un tren de pulsos coherente (PRI con estructura, AOA único).


</details>

---
## 9. Producto

In [ ]:
os.makedirs("resultados", exist_ok=True)
with pd.ExcelWriter("resultados/D4_identificacion_GMM.xlsx") as w:
    identificacion.to_excel(w, sheet_name="IDENTIFICACION", index=False)
    tabla.to_excel(w, sheet_name="FICHAS_GRUPOS")
    pd.DataFrame({"K": ks, "BIC": bics}).to_excel(w, sheet_name="BIC", index=False)
print("Guardado: resultados/D4_identificacion_GMM.xlsx ✔")

### Preguntas de repaso
1. ¿Qué ventaja concreta tiene la **probabilidad de pertenencia** del GMM frente a las ventanas manuales del Lab 4?
2. ¿Por qué no se incluye el DTOA como variable del clustering?
3. Un grupo del GMM tiene el mismo AOA y la misma PRI que otro, pero distinto PW. ¿Son dos emisores?
4. ¿Por qué la RF es criterio **excluyente** en el cotejo con la biblioteca, pero el scan no?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. Permite **cuantificar** la confianza de cada asignación, aislar pulsos ambiguos y declarar en el informe qué fracción de la evidencia es sólida.
2. Porque en un tren intercalado el DTOA entre pulsos consecutivos mezcla emisores: no es una propiedad del pulso. La PRI se calcula **dentro** de cada grupo, después de separar.
3. No necesariamente: puede ser **el mismo emisor en dos modos** (cambio de PW), o un emisor con PW variable. Hay que mirar la **línea de tiempo** (¿se alternan o se suceden?) y la PRI.
4. La RF de un radar está acotada por su hardware (magnetrón, TWT, filtros): un valor fuera de rango es casi imposible. El scan, en cambio, depende del **modo** (búsqueda, seguimiento) y de la geometría, y puede no observarse completo en una captura corta.


</details>

---
### ✅ Fin del Laboratorio 5

**Antes de continuar**, guarda tu trabajo: *Archivo → Guardar una copia en Drive* (Colab) o *File → Save* (Jupyter).

➡️ **Siguiente:** **Laboratorio 6 · Integrador: Informe Preliminar de Interceptación ELINT/ESM** (misión completa por grupos).

<span style="font-size:11px;color:#7a7a7a">Curso de Guerra Electrónica · Análisis de Señales de Radar · Material de instrucción con datos sintéticos.</span>